In [ ]:
from functools import reduce
from itertools import zip_longest

from nn import *
from ray import train, tune


def Seq(*_):
    _ = [_ for _ in _ for _ in (_ if type(_) == Sequential else [_]) if _ and not isinstance(_, Identity)]
    return Sequential(*_) if len(_) > 1 else _[0] if _ else Identity()


def add(a, b):
    return a if b is None else b if a is None else a + b


def ConvLeft(i, o, r, bias=False, **_):
    return Seq(Conv1d(i, o, r, padding=r - 1, bias=bias, **_), *[F(torch.Tensor.__getitem__, (..., slice(1 - r)))] if r > 1 else [])


def SepConvLeft(i, h, o, r=1, bias=False):
    return MinModel(
        Seq(
            Conv1d(i, h, 1, bias=False),
            MinModel(
                Seq(
                    ConvLeft(h, h, r, groups=h),
                    Conv1d(h, o, 1, bias=bias),
                ),
                ConvLeft(h, o, r, bias=bias),
            ),
        ),
        Seq(
            ConvLeft(i, h, r),
            Conv1d(h, o, 1, bias=bias),
        ),
        ConvLeft(i, o, r, bias=bias),
    )


class Zip(Sequential):
    def forward(s, x):
        return reduce(torch.add, [s(x) for s, x in zip(s, x)])


class Res(Sequential):
    def forward(s, x):
        x = [x]
        for s in s:
            x.append(s(x))
        return x[-1]


def Fullin(i, c, bias=False):
    return Zip(*[SepConvLeft(**i, o=c, bias=bias and not nonlast) for nonlast, i in enumerate(i, 1 - len(i))])


def Cov(i, c, h, o, r=1, bias=False):
    return Seq(
        Fullin(i, c, bias),
        *[
            MinModel(
                SepConvLeft(c, c, o, r),
                Seq(
                    SepConvLeft(c, c, h, r),
                    Conv1d(h, o, 1, bias=False),
                ),
            ),
            BatchNorm1d(o, track_running_stats=False),
            SiLU(True),
        ]
        if o
        else [],
    )


def Model(e, i, c=[]):
    return Seq(
        Embedding(e, i),
        F(torch.transpose, -2, -1),
        Res(*[Cov(**c, bias=not nonlast) for nonlast, c in enumerate(c, 1 - len(c))]),
    )


def sample(**_):
    return {i: getattr(_, "sample", lambda: _)() for i, _ in _.items()}


def train(
    _,
    a=("py.txt", 32, 256),
    m=("cuda", torch.bfloat16),
    O={"max_lr": 0.003, "total_steps": 10000, "pct_start": 0.01, "div_factor": 9, "final_div_factor": 99},
    compile=torch.compile,
    tqdm=lambda x: x,
    report=train.report,
    sz=25000,
):
    a = Dat(*a)
    _ = param(_)
    m = compile(sized(lambda x: Model(**arg(**_ * x)), sz)).to(*m)
    o = torch.optim.Adam(m.parameters())
    O = torch.optim.lr_scheduler.OneCycleLR(o, **O)
    X = a()
    L = A = 0
    for _ in tqdm(range(1, O.total_steps + 1)):
        x, X = X, a()
        y = m(x[:, :-1])
        x = x[:, 1:]
        l = functional.cross_entropy(y, x)
        l.backward()
        o.step()
        o.zero_grad()
        O.step()
        L += l.detach()
        A += (y.argmax(1) == x).sum()
        if not _ % 100:
            report({"l": float(L), "a": float(A) / x.numel()})
            L = A = 0

In [ ]:
tuner = tune.Tuner(
    tune.with_resources(train, {"gpu": 0.33}),
    param_space=dict(
        e_c3c=68,
        i_c0i0i_c1i0i_c2i0i_c3i0i=tune.loguniform(0.1, 4),
        c0o_c1i1i_c2i1i_c3i1i=tune.loguniform(0.1, 4),
        c1o_c2i2i_c3i2i=tune.loguniform(0.1, 4),
        c2o_c3i3i=tune.loguniform(0.1, 4),
        c0i0h=tune.loguniform(0.1, 4),
        c1i0h=tune.loguniform(0.1, 4),
        c1i1h=tune.loguniform(0.1, 4),
        c2i0h=tune.loguniform(0.1, 4),
        c2i1h=tune.loguniform(0.1, 4),
        c2i2h=tune.loguniform(0.1, 4),
        c3i0h=tune.loguniform(0.1, 4),
        c3i1h=tune.loguniform(0.1, 4),
        c3i2h=tune.loguniform(0.1, 4),
        c3i3h=tune.loguniform(0.1, 4),
        c0c=tune.loguniform(0.1, 4),
        c1c=tune.loguniform(0.1, 4),
        c2c=tune.loguniform(0.1, 4),
        c0h=tune.loguniform(0.1, 4),
        c1h=tune.loguniform(0.1, 4),
        c2h=tune.loguniform(0.1, 4),
        c0r=tune.randint(1, 7 + 1),
        c1r=tune.randint(1, 7 + 1),
        c2r=tune.randint(1, 7 + 1),
        c3h_c3o=0,
    ),
    tune_config=tune.TuneConfig(
        mode="min",
        metric="l",
        num_samples=500,
        max_concurrent_trials=3,
        scheduler=tune.schedulers.ASHAScheduler(max_t=100, grace_period=3),
    ),
)
a = tuner.fit()

In [ ]:
df = a.get_dataframe()
df = df[df.training_iteration == 100][["l", "a"] + [i for i in df.columns if "/" in i]].rename(columns=lambda x: x.split("/")[-1])[:10]
df.agg("min mean median max std".split()).T

In [ ]:
tuner2 = tune.Tuner(
    tune.with_resources(train, {"gpu": 0.33}),
    param_space=dict(
        e_c7c=68,
        i_c0i0i_c1i0i_c2i0i_c3i0i_c4i0i_c5i0i_c6i0i_c7i0i=1.0,
        c0o_c1i1i_c2i1i_c3i1i_c4i1i_c5i1i_c6i1i_c7i1i=tune.loguniform(0.1, 4),
        c1o_c2i2i_c3i2i_c4i2i_c5i2i_c6i2i_c7i2i=tune.loguniform(0.1, 4),
        c2o_c3i3i_c4i3i_c5i3i_c6i3i_c7i3i=tune.loguniform(0.1, 4),
        c3o_c4i4i_c5i4i_c6i4i_c7i4i=tune.loguniform(0.1, 4),
        c4o_c5i5i_c6i5i_c7i5i=tune.loguniform(0.1, 4),
        c5o_c6i6i_c7i6i=tune.loguniform(0.1, 4),
        c6o_c7i7i=tune.loguniform(0.1, 4),
        c0i0h=tune.loguniform(0.1, 4),
        c1i0h=tune.loguniform(0.1, 4),
        c1i1h=tune.loguniform(0.1, 4),
        c2i0h=tune.loguniform(0.1, 4),
        c2i1h=tune.loguniform(0.1, 4),
        c2i2h=tune.loguniform(0.1, 4),
        c3i0h=tune.loguniform(0.1, 4),
        c3i1h=tune.loguniform(0.1, 4),
        c3i2h=tune.loguniform(0.1, 4),
        c3i3h=tune.loguniform(0.1, 4),
        c4i0h=tune.loguniform(0.1, 4),
        c4i1h=tune.loguniform(0.1, 4),
        c4i2h=tune.loguniform(0.1, 4),
        c4i3h=tune.loguniform(0.1, 4),
        c4i4h=tune.loguniform(0.1, 4),
        c5i0h=tune.loguniform(0.1, 4),
        c5i1h=tune.loguniform(0.1, 4),
        c5i2h=tune.loguniform(0.1, 4),
        c5i3h=tune.loguniform(0.1, 4),
        c5i4h=tune.loguniform(0.1, 4),
        c5i5h=tune.loguniform(0.1, 4),
        c6i0h=tune.loguniform(0.1, 4),
        c6i1h=tune.loguniform(0.1, 4),
        c6i2h=tune.loguniform(0.1, 4),
        c6i3h=tune.loguniform(0.1, 4),
        c6i4h=tune.loguniform(0.1, 4),
        c6i5h=tune.loguniform(0.1, 4),
        c6i6h=tune.loguniform(0.1, 4),
        c7i0h=tune.loguniform(0.1, 4),
        c7i1h=tune.loguniform(0.1, 4),
        c7i2h=tune.loguniform(0.1, 4),
        c7i3h=tune.loguniform(0.1, 4),
        c7i4h=tune.loguniform(0.1, 4),
        c7i5h=tune.loguniform(0.1, 4),
        c7i6h=tune.loguniform(0.1, 4),
        c7i7h=tune.loguniform(0.1, 4),
        c0c=tune.loguniform(0.1, 4),
        c1c=tune.loguniform(0.1, 4),
        c2c=tune.loguniform(0.1, 4),
        c3c=tune.loguniform(0.1, 4),
        c4c=tune.loguniform(0.1, 4),
        c5c=tune.loguniform(0.1, 4),
        c6c=tune.loguniform(0.1, 4),
        c0h=tune.loguniform(0.1, 4),
        c1h=tune.loguniform(0.1, 4),
        c2h=tune.loguniform(0.1, 4),
        c3h=tune.loguniform(0.1, 4),
        c4h=tune.loguniform(0.1, 4),
        c5h=tune.loguniform(0.1, 4),
        c6h=tune.loguniform(0.1, 4),
        c0r=tune.randint(1, 7 + 1),
        c1r=tune.randint(1, 7 + 1),
        c2r=tune.randint(1, 7 + 1),
        c3r=tune.randint(1, 7 + 1),
        c4r=tune.randint(1, 7 + 1),
        c5r=tune.randint(1, 7 + 1),
        c6r=tune.randint(1, 7 + 1),
        c7h_c7o=0,
    ),
    tune_config=tune.TuneConfig(
        mode="min",
        metric="l",
        num_samples=10000,
        max_concurrent_trials=3,
        scheduler=tune.schedulers.ASHAScheduler(max_t=100, grace_period=2),
    ),
)
a2 = tuner2.fit()